In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, mean_squared_error
import optuna

# Rendre Optuna silencieux
optuna.logging.set_verbosity(optuna.logging.WARNING)

df = pd.read_csv('train.csv')
#####################

# --- 1. Préparation ---
categorical_features = ['type_contrat', 'utilisation', 'essence_vehicule', 'marque_vehicule', 'sex_conducteur1']

numerical_features = ['bonus', 'duree_contrat', 'age_conducteur1', 'anciennete_permis1', 'prix_vehicule', 
                      'vitesse_vehicule', 'anciennete_vehicule', 'poids_puissance']

# Remplissage simple et Feature Engineering
df['anciennete_vehicule'] = df['anciennete_vehicule'].fillna(-1)
df['poids_puissance'] = df['poids_vehicule'] / df['din_vehicule']

# --- 1. FONCTION DE FEATURE ENGINEERING ---
def apply_feature_engineering(df):
    # Ratios techniques
    df['ratio_poids_puissance'] = df['poids_vehicule'] / (df['din_vehicule'] + 1)
    df['log_prix_vehicule'] = np.log1p(df['prix_vehicule'])
    
    # Expérience
    df['age_obtention_permis'] = df['age_conducteur1'] - df['anciennete_permis1']
    df['ratio_vie_conduite'] = df['anciennete_permis1'] / (df['age_conducteur1'] - 17).replace(0, 1)
    
    # Géographie simplifiée
    df['cp_dep'] = df['code_postal'].astype(str).str.zfill(5).str[:2]
    
    # Variables binaires
    df['is_sportive'] = ((df['din_vehicule'] > 150) | (df['vitesse_vehicule'] > 200)).astype(int)
    
    return df

# Application
df = apply_feature_engineering(df)

# Mise à jour des listes
categorical_features += ['cp_dep', 'is_sportive']
numerical_features += ['ratio_poids_puissance', 'log_prix_vehicule', 'age_obtention_permis', 'ratio_vie_conduite']


# =====================================================================
# --- 2. MODÈLE FRÉQUENCE (Probabilité de sinistre) ---
# =====================================================================
X = df[numerical_features + categorical_features]
# y_freq = (df['nombre_sinistres'] > 0).astype(int)
y_freq = df['nombre_sinistres']


X_train, X_test, y_train, y_test = train_test_split(X, y_freq, test_size=0.3, random_state=8)

def objective_freq(trial):               
    params = {
        "iterations": trial.suggest_int("iterations", 100, 600),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "random_strength": trial.suggest_float("random_strength", 1e-9, 10, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "eval_metric": "AUC",
        "verbose": False
    }
    
    model = CatBoostClassifier(**params, auto_class_weights="Balanced")
    model.fit(X_train, y_train, cat_features=categorical_features, eval_set=(X_test, y_test), early_stopping_rounds=20)
    
    preds_proba = model.predict_proba(X_test)[:, 1]
    auc_val = roc_auc_score(y_test, preds_proba)
    
    print(f'Freq parameters: {params}, AUC: {auc_val:.4f}')
    return auc_val

print("--- Recherche des hyperparamètres FRÉQUENCE (Maximisation AUC) ---")
study_freq = optuna.create_study(direction="maximize")
study_freq.optimize(objective_freq, n_trials=200) 

# Entraînement final Fréquence
print(f"\nBest Freq parameters: {study_freq.best_params}")
model_freq = CatBoostClassifier(**study_freq.best_params, auto_class_weights="Balanced", eval_metric="AUC", verbose=False)
model_freq.fit(X_train, y_train, cat_features=categorical_features, eval_set=(X_test, y_test))

# =====================================================================
# --- 3. MODÈLE SÉVÉRITÉ (Montant moyen) ---
# =====================================================================
# Uniquement sur les lignes avec un montant > 0
# df_claims = df[df['montant_sinistre'] > 0].copy()
df_claims = df.copy()  # Pour éviter les SettingWithCopyWarning
X_sev = df_claims[numerical_features + categorical_features]
y_sev = df_claims['montant_sinistre']

# Split spécifique pour la sévérité
X_train_sev, X_test_sev, y_train_sev, y_test_sev = train_test_split(X_sev, y_sev, test_size=0.3, random_state=8)

def objective_sev(trial):               
    params = {
        "iterations": trial.suggest_int("iterations", 100, 600),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "random_strength": trial.suggest_float("random_strength", 1e-9, 10, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "objective": 'Tweedie:variance_power=1.5', # On garde l'objectif Tweedie très bon pour l'assurance
        "verbose": False
    }
    
    model = CatBoostRegressor(**params) # Pas de class_weights pour une régression
    model.fit(X_train_sev, y_train_sev, cat_features=categorical_features, eval_set=(X_test_sev, y_test_sev), early_stopping_rounds=20)
    
    preds = model.predict_proba(X_test_sev)
    mse_val = mean_squared_error(y_test_sev, preds)
    
    print(f'Sev parameters: {params}, MSE: {mse_val:.2f}')
    return mse_val

print("\n--- Recherche des hyperparamètres SÉVÉRITÉ (Minimisation MSE) ---")
study_sev = optuna.create_study(direction="minimize")
study_sev.optimize(objective_sev, n_trials=200) # Ajuster si besoin

# Entraînement final Sévérité
print(f"\nBest Sev parameters: {study_sev.best_params}")
# On s'assure de réinjecter l'objectif Tweedie qui n'est pas dans best_params par défaut s'il n'était pas dans l'espace de recherche suggest
best_sev_params = study_sev.best_params
best_sev_params['objective'] = 'Tweedie:variance_power=1.5'

model_sev = CatBoostRegressor(**best_sev_params, verbose=False)
model_sev.fit(X_train_sev, y_train_sev, cat_features=categorical_features, eval_set=(X_test_sev, y_test_sev))


# =====================================================================
# --- MÉTRIQUES FINALES ---
# =====================================================================
y_pred_freq_proba = model_freq.predict_proba(X_test)[:, 1]
auc_final = roc_auc_score(y_test, y_pred_freq_proba)
print(f"\nROC AUC final for the frequency model: {auc_final:.4f}")

y_pred_sev = model_sev.predict(X_test_sev)
mse_sev_final = mean_squared_error(y_test_sev, y_pred_sev)
print(f"Mean Squared Error final for the severity model: {mse_sev_final:.2f}")

# --- 4. CALCUL DU COÛT ESPÉRÉ SUR TOUT LE DATASET ---
df['probabilite'] = model_freq.predict_proba(X)[:, 1]
df['montant_potentiel'] = model_sev.predict(X)
df['pred'] = df['probabilite'] * df['montant_potentiel']

print("\nAperçu des prédictions :")
print(df[['probabilite', 'montant_potentiel', 'pred']].head())



--- Recherche des hyperparamètres FRÉQUENCE (Maximisation AUC) ---
Freq parameters: {'iterations': 306, 'learning_rate': 0.053798624011988615, 'depth': 5, 'l2_leaf_reg': 6.5093625313682395, 'random_strength': 3.756135846657638e-06, 'bagging_temperature': 0.25910028577771504, 'border_count': 184, 'eval_metric': 'AUC', 'verbose': False}, AUC: 0.6547
Freq parameters: {'iterations': 140, 'learning_rate': 0.032937693805679444, 'depth': 8, 'l2_leaf_reg': 2.5454765634512544, 'random_strength': 1.4680720825792173, 'bagging_temperature': 0.7894760194406033, 'border_count': 177, 'eval_metric': 'AUC', 'verbose': False}, AUC: 0.6576
Freq parameters: {'iterations': 523, 'learning_rate': 0.08650418499743441, 'depth': 7, 'l2_leaf_reg': 9.039626762098655, 'random_strength': 7.808042983417962e-08, 'bagging_temperature': 0.9744358714688338, 'border_count': 42, 'eval_metric': 'AUC', 'verbose': False}, AUC: 0.6534
Freq parameters: {'iterations': 398, 'learning_rate': 0.020713237873803865, 'depth': 6, 'l2_

In [37]:
# 1. Charger le nouveau fichier
nouveau_df = pd.read_csv('test.csv')

# 2. Prétraitement identique (très important !)
# On remplit les valeurs manquantes comme pour l'entraînement
nouveau_df['anciennete_vehicule'] = nouveau_df['anciennete_vehicule'].fillna(nouveau_df['anciennete_vehicule'].median())
nouveau_df['poids_puissance'] = nouveau_df['poids_vehicule'] / nouveau_df['din_vehicule']
# 3. Sélectionner les colonnes utilisées par le modèle
# (Assurez-vous que numerical_features et categorical_features sont bien définis)
X_nouveau = nouveau_df[numerical_features + categorical_features]

# 4. Appliquer les prédictions
# Probabilité de sinistre (Modèle 1)
nouveau_df['probabilite'] = model_freq.predict_proba(X_nouveau)[:, 1]

# Montant estimé si sinistre il y a (Modèle 2)
nouveau_df['montant_potentiel'] = model_sev.predict(X_nouveau)

# Coût espéré (Prime pure)
nouveau_df['pred'] = nouveau_df['probabilite'] * nouveau_df['montant_potentiel']

# 5. Sauvegarder les résultats
nouveau_df.to_csv('resultats_predictions.csv', index=False)
print("Les prédictions ont été sauvegardées dans 'resultats_predictions.csv'")


pp=pd.read_csv(f'resultats_predictions.csv')
df_final = pp[['index', 'pred']]

# 3. Sauvegarder le résultat dans un nouveau fichier CSV
# index=False permet de ne pas rajouter une colonne de numérotation inutile
df_final.to_csv(f'resultat_final.csv', index=False)

print(f'Le fichier filtré a été sauvegardé sous le nom resultat_final.csv')

Les prédictions ont été sauvegardées dans 'resultats_predictions.csv'
Le fichier filtré a été sauvegardé sous le nom resultat_final.csv


In [7]:
# all = np.linspace(1.2, 1.3, 10)
# for i in all:
#     # 1. Charger le nouveau fichier
#     nouveau_df = pd.read_csv('test.csv')

#     # 2. Prétraitement identique (très important !)
#     # On remplit les valeurs manquantes comme pour l'entraînement
#     nouveau_df['anciennete_vehicule'] = nouveau_df['anciennete_vehicule'].fillna(nouveau_df['anciennete_vehicule'].median())

#     # 3. Sélectionner les colonnes utilisées par le modèle
#     # (Assurez-vous que numerical_features et categorical_features sont bien définis)
#     X_nouveau = nouveau_df[numerical_features + categorical_features]

#     # 4. Appliquer les prédictions
#     # Probabilité de sinistre (Modèle 1)
#     nouveau_df['probabilite'] = model_freq.predict_proba(X_nouveau)[:, 1]

#     # Montant estimé si sinistre il y a (Modèle 2)
#     nouveau_df['montant_potentiel'] = model_sev.predict(X_nouveau)

#     # Coût espéré (Prime pure)
#     nouveau_df['pred'] = nouveau_df['probabilite'] * nouveau_df['montant_potentiel']

#     # 5. Sauvegarder les résultats
#     nouveau_df.to_csv('resultats_predictions.csv', index=False)
#     print("Les prédictions ont été sauvegardées dans 'resultats_predictions.csv'")


#     pp=pd.read_csv(f'resultats_predictions.csv')
#     df_final = pp[['index', 'pred']]

#     # 3. Sauvegarder le résultat dans un nouveau fichier CSV
#     # index=False permet de ne pas rajouter une colonne de numérotation inutile
#     df_final.to_csv(f'resultat_final2_{i}.csv', index=False)

#     print(f'Le fichier filtré a été sauvegardé sous le nom resultat_final2_{i}.csv')

In [8]:
# import optuna
# from catboost import CatBoostClassifier, Pool
# from sklearn.metrics import f1_score
# from sklearn.model_selection import train_test_split


# # Définition de la fonction objectif
# def objective(trial):
#     # Liste des hyperparamètres à tester
#     param = {
#         "iterations": trial.suggest_int("iterations", 100, 1000),
#         "depth": trial.suggest_int("depth", 4, 10),
#         "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
#         "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
#         "random_strength": trial.suggest_float("random_strength", 1, 10),
#         "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 1),
#         "border_count": trial.suggest_int("border_count", 32, 255),
#         "auto_class_weights": "Balanced", # Très important pour ton problème de fréquence
#         "verbose": False,
#         "eval_metric": "F1",
#         "task_type": "CPU"
#     }

#     # Création et entraînement du modèle
#     model = CatBoostClassifier(**param)
#     model.fit(
#         X_train, y_train,
#         cat_features=categorical_features,
#         eval_set=(X_valid, y_valid),
#         early_stopping_rounds=50
#     )

#     # On prédit sur le set de validation pour évaluer
#     preds = model.predict(X_valid)
#     score = f1_score(y_valid, preds)
    
#     return score # Optuna va chercher à maximiser ce score

# # 2. Lancement de l'étude
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=50) # On teste 50 combinaisons différentes

# # 3. Affichage des meilleurs résultats
# print("Meilleurs hyperparamètres :", study.best_params)
# print("Meilleur score F1 :", study.best_value)